# Лабораторная работа № 5 — вариант 4

**Источник:** `https://quotes.toscrape.com/tag/...`  
**Маршрут:** каталог → страницы тегов → страницы авторов.  
**Проект:** `lab5_variant4/quotes_crawler`.

На площадке 100 цитат, но три цитаты не имеют тегов и недоступны из точки
входа варианта 4. В теговом обходе получено 97 цитат и 232 связи
«цитата–тег»: всего 329 строк нормализованной SQLite-базы.

## 1. Окружение и структура проекта

In [1]:
import json
import sqlite3
import subprocess
import sys
from pathlib import Path

import pandas as pd
import scrapy
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)

LAB_DIR = Path.cwd()
PROJECT_DIR = LAB_DIR / "lab5_variant4"
DATA_DIR = PROJECT_DIR / "data"
DB_PATH = DATA_DIR / "crawl.db"

print("Python:", sys.executable)
print("Scrapy:", scrapy.__version__)
print("Проект:", PROJECT_DIR)
print("Основные файлы:")
for path in sorted(PROJECT_DIR.rglob("*.py")):
    if "__pycache__" not in path.parts:
        print("-", path.relative_to(PROJECT_DIR))

Python: C:\Users\k_dys\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe
Scrapy: 2.19.0
Проект: E:\Labs_2_course\Data_mining\lab_5\lab5_variant4
Основные файлы:
- quality\smoke_check.py
- quotes_crawler\__init__.py
- quotes_crawler\items.py
- quotes_crawler\middlewares.py
- quotes_crawler\pipelines.py
- quotes_crawler\settings.py
- quotes_crawler\spiders\__init__.py
- quotes_crawler\spiders\quotes_tags.py


Проект был создан штатной командой:

```powershell
python3.10 -m scrapy startproject quotes_crawler lab5_variant4
```

Код разделён по ответственности:

- `items.py` — схема `QuoteItem`;
- `spiders/quotes_tags.py` — переходы и селекторы;
- `pipelines.py` — валидация, очистка, дедупликация и SQLite;
- `settings.py` — вежливость и предохранители.

## 2. Команды запуска

Ноутбук читает уже выполненные прогоны. Чтобы повторить их, переключите
`RUN_CRAWLS` в `True`. Перед повтором демонстраций следует удалить только их
собственные БД, JSONL, cache/jobstate — точные команды приведены в README.

In [2]:
RUN_CRAWLS = False


def run_crawl(arguments, timeout=900):
    command = [sys.executable, "-m", "scrapy", "crawl", "quotes_tags", *arguments]
    result = subprocess.run(
        command,
        cwd=PROJECT_DIR,
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-4000:])
        raise RuntimeError(f"Scrapy завершился с кодом {result.returncode}")
    return result


final_arguments = [
    "-a", "run_id=final",
    "-s", "SQLITE_PATH=data/crawl.db",
    "-s", "HTTPCACHE_DIR=httpcache_final_network_v4",
    "-s", "LOG_LEVEL=ERROR",
    "-O", "data/items.jsonl",
]

if RUN_CRAWLS:
    run_crawl(final_arguments)
else:
    print("Повторный краулинг отключён; используются сохранённые результаты.")
    print("Команда итогового запуска:")
    print(sys.executable, "-m scrapy crawl quotes_tags", " ".join(final_arguments))

Повторный краулинг отключён; используются сохранённые результаты.
Команда итогового запуска:
C:\Users\k_dys\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m scrapy crawl quotes_tags -a run_id=final -s SQLITE_PATH=data/crawl.db -s HTTPCACHE_DIR=httpcache_final_network_v4 -s LOG_LEVEL=ERROR -O data/items.jsonl


## 3. Итоговая статистика Scrapy

In [3]:
final_stats = json.loads((DATA_DIR / "stats_final.json").read_text(encoding="utf-8"))

stat_keys = [
    "close_reason",
    "elapsed_wall_seconds",
    "catalog/pages",
    "tag/pages",
    "item_scraped_count",
    "item_dropped_count",
    "dropped/duplicate",
    "db/quotes_written",
    "db/tag_links_written",
    "db/batches",
    "downloader/request_count",
    "downloader/response_bytes",
    "dupefilter/filtered",
    "httpcache/hit",
    "httpcache/store",
    "retry/count",
    "network/errors",
]

stats_table = pd.DataFrame(
    [{"metric": key, "value": final_stats.get(key, 0)} for key in stat_keys]
)
display(stats_table)

status_rows = [
    {
        "status": int(key.rsplit("/", 1)[-1]),
        "responses": value,
    }
    for key, value in final_stats.items()
    if key.startswith("downloader/response_status_count/")
]
display(pd.DataFrame(status_rows).sort_values("status"))

,metric,value
0,close_reason,finished
1,elapsed_wall_seconds,62.059
2,catalog/pages,10
3,tag/pages,142
4,item_scraped_count,97
5,item_dropped_count,135
6,dropped/duplicate,135
7,db/quotes_written,97
8,db/tag_links_written,232
9,db/batches,5


,status,responses
1,200,384
0,404,1


`CLOSESPIDER_ITEMCOUNT = 300` присутствует как верхний предохранитель, но не
срабатывает: источник даёт только 97 уникальных цитат после обязательной
дедупликации. Поэтому `close_reason = finished` означает исчерпание очереди.

## 4. Качество нормализованной базы

In [4]:
with sqlite3.connect(DB_PATH) as connection:
    quotes = pd.read_sql_query("SELECT * FROM quotes", connection)
    quote_tags = pd.read_sql_query("SELECT * FROM quote_tags", connection)

summary = pd.DataFrame(
    [
        {"table": "quotes", "rows": len(quotes)},
        {"table": "quote_tags", "rows": len(quote_tags)},
        {"table": "total", "rows": len(quotes) + len(quote_tags)},
    ]
)
display(summary)

print("Уникальных бизнес-ключей:", quotes["business_key"].nunique())
print("Дубликатов бизнес-ключа:", quotes.duplicated("business_key").sum())
print("Дубликатов связи цитата–тег:", quote_tags.duplicated(["business_key", "tag"]).sum())

assert len(quotes) == 97
assert len(quote_tags) == 232
assert len(quotes) + len(quote_tags) >= 300
assert quotes["business_key"].is_unique

,table,rows
0,quotes,97
1,quote_tags,232
2,total,329


Уникальных бизнес-ключей: 97
Дубликатов бизнес-ключа: 0
Дубликатов связи цитата–тег: 0


In [5]:
required = [
    "quote",
    "author",
    "author_url",
    "born_date",
    "born_location",
    "biography",
]
completeness = pd.DataFrame(
    {
        "field": required,
        "complete_percent": [
            round(
                100
                * (
                    quotes[field].notna()
                    & quotes[field].astype(str).str.strip().ne("")
                ).mean(),
                2,
            )
            for field in required
        ],
    }
)
display(completeness)
assert completeness["complete_percent"].min() >= 95

,field,complete_percent
0,quote,100.0
1,author,100.0
2,author_url,100.0
3,born_date,100.0
4,born_location,100.0
5,biography,100.0


## 5. Аналитические агрегаты

In [6]:
with sqlite3.connect(DB_PATH) as connection:
    top_tags = pd.read_sql_query(
        """
        SELECT tag, COUNT(*) AS quotes_count
        FROM quote_tags
        GROUP BY tag
        ORDER BY quotes_count DESC, tag
        LIMIT 12
        """,
        connection,
    )
    author_stats = pd.read_sql_query(
        """
        SELECT q.author,
               COUNT(DISTINCT q.business_key) AS quotes_count,
               COUNT(qt.tag) AS tag_links,
               ROUND(AVG(LENGTH(q.quote)), 1) AS avg_quote_length
        FROM quotes q
        LEFT JOIN quote_tags qt USING (business_key)
        GROUP BY q.author
        ORDER BY quotes_count DESC, q.author
        LIMIT 10
        """,
        connection,
    )

display(top_tags)
display(author_stats)

length_stats = pd.DataFrame(
    {
        "metric": ["quote_length", "biography_length"],
        "mean": [quotes["quote"].str.len().mean(), quotes["biography"].str.len().mean()],
        "median": [quotes["quote"].str.len().median(), quotes["biography"].str.len().median()],
        "max": [quotes["quote"].str.len().max(), quotes["biography"].str.len().max()],
    }
)
display(length_stats.round(1))

,tag,quotes_count
0,love,14
1,inspirational,13
2,life,13
3,humor,12
4,books,11
5,reading,7
6,friendship,5
7,friends,4
8,truth,4
9,attributed-no-source,3


,author,quotes_count,tag_links,avg_quote_length
0,Albert Einstein,10,25,93.0
1,J.K. Rowling,8,11,103.2
2,Dr. Seuss,6,11,102.5
3,Marilyn Monroe,6,13,552.4
4,Mark Twain,6,14,72.6
5,C.S. Lewis,5,13,128.2
6,Jane Austen,5,15,154.2
7,Bob Marley,3,3,284.3
8,Charles Bukowski,2,2,159.5
9,Eleanor Roosevelt,2,4,50.0


,metric,mean,median,max
0,quote_length,120.8,85.0,1082
1,biography_length,2199.0,1967.0,3830


## 6. HTTP-кэш

Два одинаковых ограниченных прогона выполнены с одним `HTTPCACHE_DIR`.

In [7]:
cache_rows = []
for run_id in ("cache1", "cache2"):
    stats = json.loads((DATA_DIR / f"stats_{run_id}.json").read_text(encoding="utf-8"))
    cache_rows.append(
        {
            "run": run_id,
            "seconds": stats.get("elapsed_wall_seconds", 0),
            "items": stats.get("item_scraped_count", 0),
            "requests": stats.get("downloader/request_count", 0),
            "httpcache_store": stats.get("httpcache/store", 0),
            "httpcache_hit": stats.get("httpcache/hit", 0),
        }
    )

cache_comparison = pd.DataFrame(cache_rows)
display(cache_comparison)

,run,seconds,items,requests,httpcache_store,httpcache_hit
0,cache1,18.079,23,66,58,8
1,cache2,8.654,28,56,23,33


Второй запуск быстрее, потому что значительная часть ответов взята из локального
HTTP-кэша. Небольшое различие количества items допустимо: `CLOSESPIDER_ITEMCOUNT`
останавливает асинхронный Scrapy, когда часть запросов уже находится в работе.

## 7. JOBDIR: остановка и продолжение

In [8]:
job_rows = []
for run_id in ("job1", "job2"):
    stats = json.loads((DATA_DIR / f"stats_{run_id}.json").read_text(encoding="utf-8"))
    job_rows.append(
        {
            "run": run_id,
            "close_reason": stats.get("close_reason"),
            "seconds": stats.get("elapsed_wall_seconds", 0),
            "items": stats.get("item_scraped_count", 0),
            "duplicates": stats.get("dropped/duplicate", 0),
            "dupefilter_filtered": stats.get("dupefilter/filtered", 0),
            "disk_enqueued": stats.get("scheduler/enqueued/disk", 0),
            "disk_dequeued": stats.get("scheduler/dequeued/disk", 0),
        }
    )

display(pd.DataFrame(job_rows))

job_files = sorted(path.name for path in (PROJECT_DIR / "jobstate").iterdir())
seen_path = PROJECT_DIR / "jobstate" / "requests.seen"
seen_fingerprints = len(seen_path.read_bytes().splitlines())
print("Файлы JOBDIR:", job_files)
print("Сохранённых отпечатков запросов:", seen_fingerprints)

with sqlite3.connect(DATA_DIR / "job.db") as connection:
    resumed_quotes = connection.execute("SELECT COUNT(*) FROM quotes").fetchone()[0]
print("Цитат в БД после двух запусков:", resumed_quotes)
assert resumed_quotes > 22

,run,close_reason,seconds,items,duplicates,dupefilter_filtered,disk_enqueued,disk_dequeued
0,job1,closespider_itemcount,17.716,25,6,74,171,64
1,job2,closespider_itemcount,16.928,23,22,21,72,91


Файлы JOBDIR: ['requests.queue', 'requests.seen', 'spider.state']
Сохранённых отпечатков запросов: 25
Цитат в БД после двух запусков: 44


Первый запуск завершён предохранителем `closespider_itemcount`, но очередь и
dupefilter остались в `jobstate`. Второй запуск с тем же JOBDIR продолжил
очередь, а не начал обход с нуля.

## 8. Smoke-check

In [9]:
smoke = subprocess.run(
    [
        sys.executable,
        "quality/smoke_check.py",
        "--db",
        "data/crawl.db",
    ],
    cwd=PROJECT_DIR,
    text=True,
    capture_output=True,
    check=True,
)
print(smoke.stdout)
smoke_result = json.loads((PROJECT_DIR / "quality/current_summary.json").read_text(encoding="utf-8"))
assert smoke_result["warnings"] == []

{
  "quotes": 97,
  "tag_links": 232,
  "total_rows": 329,
  "empty_required_share": 0.0,
  "warnings": []
}


## 9. Риски редизайна

| Селектор | Назначение | Риск | Раннее обнаружение |
|---|---|---|---|
| `.quote` | контейнер цитаты | переименование класса | падение числа записей в smoke-check |
| `.text::text` | текст цитаты | изменение вложенности | рост `dropped/missing_fields` |
| `a[href*='/author/']` | ссылка автора | изменение URL | `dropped/missing_author_link` |
| `.author-born-date::text` | дата рождения | изменение карточки автора | `dropped/missing_fields` или `dropped/bad_date` |
| `li.next a::attr(href)` | пагинация | изменение навигации | меньше 10 страниц и падение общего объёма |

Индексные XPath не используются. Smoke-check предупреждает при падении числа
строк более чем на 20% или доле пустых обязательных полей выше 5%.

## 10. Вывод

Scrapy обеспечивает штатную структуру проекта, асинхронный двухуровневый
обход, вежливое регулирование нагрузки, повтор запросов, pipeline, статистику,
HTTP-кэш и возобновление JOBDIR. Физическое ограничение источника отражено
честно: результат варианта 4 — 97 тегированных цитат и 232 связи, а не
искусственно размноженные 300 цитат.